# Preparação das áreas ardidas EFFIS — Região Centro

Este notebook reproduz os produtos para os cenários **C5 e C6**.

In [ ]:
import sys
from pathlib import Path

sys.path.append("/code/scripts")

from aoi_utils import check_alignment
from burned_area_utils import (
    align_outputs_effis,
    create_binary_raster,
    create_count_raster,
    prepare_effis
)

## Configuração

In [ ]:
# ALTERAR APENAS ESTA CÉLULA

area = "extremadura" #opções: "centro" ou "extremadura"
source = "effis"

train_start = 2008
train_end = 2024
validation_year = 2025
min_area_ha = 5

raw_dir = "/code/pen/12f0c2f73c514e8b905d6f028efd81c1"
field_date = "initialdat"
field_area = "area_ha"

base = f"/code/data/processed/{area}"
effis_dir = f"{base}/area_ardida/{source}"

aoi = f"{base}/aoi/{area}.shp"
reference = f"{base}/topo/derived/dem_{area}_clip.tif"

out_year = f"{effis_dir}/yearly_vector"
out_aoi = f"{effis_dir}/yearly_aoi"
out_train = f"{effis_dir}/train"
out_valid = f"{effis_dir}/valid"
out_count = f"{effis_dir}/raster_count"
out_binary = f"{effis_dir}/raster_binary"
out_temp = f"/code/data/scratch/tmp_periods/{area}/area_ardida/{source}"

## Preparação dos vetores anuais

In [ ]:
summary = prepare_effis(
    raw_dir=raw_dir,
    aoi=aoi,
    reference=reference,
    yearly_folder=out_year,
    clipped_folder=out_aoi,
    train_folder=out_train,
    valid_folder=out_valid,
    field_date=field_date,
    field_area=field_area,
    start_year=train_start,
    end_year=validation_year,
    validation_year=validation_year,
    min_area_ha=min_area_ha
)

summary

In [ ]:
print("Anuais separados:", len(list(Path(out_year).glob("aa_*.shp"))))
print("Anuais recortados:", len(list(Path(out_aoi).glob("aa_*.shp"))))
print("Treino:", len(list(Path(out_train).glob("aa_*.shp"))))
print("Validação:", len(list(Path(out_valid).glob("aa_*.shp"))))

## Períodos temporais de C5 e C6

In [ ]:
periods = {
    "2008_2024": (range(2008, 2025), out_train),
    "2025": ([2025], out_valid)
}


# Região Centro — períodos associados à COS
if area == "centro":
    periods.update({
        "2008_2009": (range(2008, 2010), out_aoi),
        "2010_2014": (range(2010, 2015), out_aoi),
        "2015_2017": (range(2015, 2018), out_aoi),
        "2018_2024": (range(2018, 2025), out_aoi),
    })


# Extremadura — períodos associados ao SIOSE e SIOSE AR
elif area == "extremadura":
    periods.update({
        "2008": ([2008], out_aoi),
        "2009_2010": (range(2009, 2011), out_aoi),
        "2011_2013": (range(2011, 2014), out_aoi),
        "2014_2016": (range(2014, 2017), out_aoi),
        "2017_2019": (range(2017, 2020), out_aoi),
        "2020_2024": (range(2020, 2025), out_aoi),
    })

## Rasters de contagem

In [ ]:
count_rasters = []

for name, (years, folder) in periods.items():
    output = f"{out_count}/rst_ba_{name}.tif"

    create_count_raster(
        years=years,
        source_folder=folder,
        temp_folder=f"{out_temp}/{name}",
        reference=reference,
        output=output
    )

    count_rasters.append(output)
    print("Criado:", output)

## Rasters binários

In [ ]:
binary_periods = ["2008_2024", "2025"]
binary_rasters = []

for name in binary_periods:
    source_raster = f"{out_count}/rst_ba_{name}.tif"
    output = f"{out_binary}/rst_ba_{name}_bin.tif"

    create_binary_raster(source_raster, output)
    binary_rasters.append(output)
    print("Criado:", output)

## Alinhamento e verificação final

In [ ]:
rasters = count_rasters + binary_rasters

align_outputs_effis(rasters, reference)
check_alignment(rasters, reference)

In [ ]:
if area == "centro":
    expected_count = {
        "rst_ba_2008_2024.tif",
        "rst_ba_2008_2009.tif",
        "rst_ba_2010_2014.tif",
        "rst_ba_2015_2017.tif",
        "rst_ba_2018_2024.tif",
        "rst_ba_2025.tif",
    }

elif area == "extremadura":
    expected_count = {
        "rst_ba_2008_2024.tif",
        "rst_ba_2008.tif",
        "rst_ba_2009_2010.tif",
        "rst_ba_2011_2013.tif",
        "rst_ba_2014_2016.tif",
        "rst_ba_2017_2019.tif",
        "rst_ba_2020_2024.tif",
        "rst_ba_2025.tif",
    }

expected_binary = {
    "rst_ba_2008_2024_bin.tif",
    "rst_ba_2025_bin.tif",
}

assert {Path(path).name for path in count_rasters} == expected_count
assert {Path(path).name for path in binary_rasters} == expected_binary

print(f"Todos os produtos EFFIS de {area} foram criados.")